# atml-pa3-t3-smoke

Stage: `smoke`

Self-contained ATML PA3 Task 3 runner. Source is embedded as a zip. Artifacts go to `/kaggle/working/results`. Internet is required for Qwen2.5-3B-Instruct and (if needed) MATH problems from HuggingFace.


In [ ]:
import os, sys, shutil, pathlib, zipfile, io, base64
STAGE = 'smoke'  # smoke|live|offline|gapfill
dst = pathlib.Path('/kaggle/working/task3')
if dst.exists():
    shutil.rmtree(dst)
dst.mkdir(parents=True)
print('Extracting embedded code zip')
CODE_B64 = '''...'''
raw = base64.b64decode(CODE_B64)
with zipfile.ZipFile(io.BytesIO(raw)) as zf:
    zf.extractall(dst)
assert (dst / 'src').is_dir() and (dst / 'configs').is_dir(), list(dst.iterdir())
os.chdir(dst)
sys.path.insert(0, str(dst))
print('CODE_ROOT', dst)
print('STAGE', STAGE)


In [ ]:
import torch, subprocess, sys
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), 'VRAM_GiB', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
pkgs = [
    'transformers>=4.45.0', 'accelerate', 'sentencepiece', 'protobuf',
    'pyyaml', 'safetensors', 'tqdm', 'pandas', 'pyarrow', 'matplotlib',
    'huggingface_hub', 'datasets', 'math-verify', 'latex2sympy2_extended', 'scipy',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade'] + pkgs)


In [ ]:
import os, sys, subprocess, pathlib, json, time, traceback, shutil
os.environ['ATML_ARTIFACTS_ROOT'] = '/kaggle/working/results'
os.environ['HF_HOME'] = '/tmp/hf_home'
os.environ['HF_HUB_CACHE'] = '/tmp/hf_home'
os.environ['TRANSFORMERS_CACHE'] = '/tmp/hf_home'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import zipfile
def _parquet_ok(root):
    root = pathlib.Path(root)
    needed = [
        root / 'qwen25-math' / 'generations.parquet',
        root / 'qwen25-math' / 'probability_metrics.parquet',
        root / 'qwen25vl-mathvista' / 'generations.parquet',
        root / 'qwen25vl-mathvista' / 'probability_metrics.parquet',
        root / 'qwen25vl-mathvista' / 'attention_metrics.parquet',
    ]
    ok = True
    for p in needed:
        sz = p.stat().st_size if p.exists() else -1
        print('check', p, 'bytes', sz, flush=True)
        if sz < 1_000_000:
            ok = False
    return ok
def _extract_zips(zips, out):
    if out.exists():
        shutil.rmtree(out)
    out.mkdir(parents=True)
    for zpath in zips:
        print('Extracting', zpath, 'size', zpath.stat().st_size, flush=True)
        with zipfile.ZipFile(zpath) as zf:
            zf.extractall(out)
    # Nested ATML-PA3-Data/ or per-folder zips from kaggle -r zip.
    nested = out / 'ATML-PA3-Data'
    if nested.exists() and (nested / 'qwen25-math').exists():
        return nested
    for sub in list(out.glob('*.zip')):
        target = out / sub.stem
        target.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(sub) as zf:
            zf.extractall(target)
    if (out / 'qwen25-math').exists():
        return out
    return out
data_candidates = [
    pathlib.Path('/kaggle/input/atml-pa3-data'),
    pathlib.Path('/kaggle/input/datasets/mtayyabtahir/atml-pa3-data'),
]
resolved = None
for cand in data_candidates:
    if not cand.exists():
        continue
    print('candidate', cand, 'children', sorted(p.name for p in cand.iterdir())[:30], flush=True)
    zips = sorted({*cand.glob('*.zip'), *cand.rglob('ATML-PA3-Data.zip'), *cand.rglob('qwen25*.zip')})
    # Prefer extracting archives; Kaggle sometimes mounts broken 0-byte parquet stubs beside the zip.
    if zips:
        resolved = _extract_zips(zips, pathlib.Path('/kaggle/working/ATML-PA3-Data'))
    elif _parquet_ok(cand):
        resolved = cand
    elif (cand / 'ATML-PA3-Data').exists() and _parquet_ok(cand / 'ATML-PA3-Data'):
        resolved = cand / 'ATML-PA3-Data'
    if resolved is not None:
        if not _parquet_ok(resolved):
            print('INVALID data root (parquet too small/missing):', resolved, flush=True)
            resolved = None
            continue
        break
if resolved is not None:
    os.environ['ATML_PA3_DATA'] = str(resolved)
    print('ATML_PA3_DATA', resolved, flush=True)
else:
    raise SystemExit('ATML-PA3-Data not found or parquet files invalid under /kaggle/input')
if STAGE == 'smoke':
    os.environ['ATML_SMOKE'] = '1'
pathlib.Path('/kaggle/working/results').mkdir(parents=True, exist_ok=True)
t0 = time.time()
module = {
    'offline': 'src.run_offline',
    'gapfill': 'src.run_gapfill',
    'live': 'src.run_live',
    'smoke': 'src.run_live',
}.get(STAGE, 'src.run_live')
cmd = [sys.executable, '-m', module, '--artifacts-root', '/kaggle/working/results']
if STAGE == 'smoke':
    cmd.append('--smoke')
if STAGE in {'offline', 'gapfill', 'live'}:
    # Require official data; do not silently fall back for live.
    print('ATML_PA3_DATA', os.environ.get('ATML_PA3_DATA'), flush=True)
print('Running', cmd, flush=True)
try:
    rc = subprocess.call(cmd)
except Exception:
    pathlib.Path('/kaggle/working/results/failure.txt').write_text(traceback.format_exc())
    raise
print('RC', rc, 'elapsed_min', round((time.time() - t0) / 60, 2))
status = {'stage': STAGE, 'returncode': rc, 'elapsed_sec': time.time() - t0}
pathlib.Path('/kaggle/working/results/stage_status.json').write_text(json.dumps(status, indent=2))
shutil.rmtree('/kaggle/working/task3', ignore_errors=True)
shutil.rmtree('/tmp/hf_home', ignore_errors=True)
if rc != 0:
    raise SystemExit(rc)


In [ ]:
import pathlib
root = pathlib.Path('/kaggle/working/results')
files = sorted(str(p.relative_to(root)) for p in root.rglob('*') if p.is_file())
print('artifact_files', len(files))
for path in files[:200]:
    print(path)
status_md = root / 'STATUS.md'
print(status_md.read_text()[:4000] if status_md.exists() else 'no STATUS.md')
summary = root / 'live' / 'live_summary.json'
print(summary.read_text()[:2000] if summary.exists() else 'no live_summary')
status = root / 'stage_status.json'
print(status.read_text() if status.exists() else 'no status')
